In [0]:

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark.sql("CREATE SCHEMA IF NOT EXISTS shoplive.gold")

events_df = spark.table("shoplive.silver.events")
orders_df = spark.table("shoplive.silver.orders")
customers_df = spark.table("shoplive.silver.customers")
products_df = spark.table("shoplive.silver.products")

orders_products_df = (
    orders_df.alias("o")
    .join(broadcast(products_df.alias("p")), "product_id")
    .select(
        F.col("o.*"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.unit_cost")
    )
)

print(f"Events: {events_df.count():,} | Orders: {orders_df.count():,} | Customers: {customers_df.count():,} | Products: {products_df.count():,}")
print(f"Orders+Products: {orders_products_df.count():,} rows")

In [0]:
customer_gold = (
    customers_df
    .join(
        orders_df.groupBy("customer_id").agg(
            F.count("order_id").alias("total_orders"),
            F.sum("quantity").alias("total_items"),
            F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_spend"),
            F.round(F.avg(F.col("quantity") * F.col("unit_price")), 2).alias("average_order_value"),
            F.min("order_date").alias("first_order_date"),
            F.max("order_date").alias("last_order_date")
        ),
        "customer_id", "left"
    )
    .select(
        "customer_id", "first_name", "second_name", "email", 
        "city", "country", "signup_date", "signup_channel",
        F.coalesce("total_orders", F.lit(0)).alias("total_orders"),
        F.coalesce("total_items", F.lit(0)).alias("total_items"),
        F.coalesce("total_spend", F.lit(0.0)).alias("total_spend"),
        "average_order_value", "first_order_date", "last_order_date",
        F.coalesce(F.datediff(F.col("last_order_date"), F.col("first_order_date")), F.lit(0)).alias("customer_lifetime_days")
    )
)

customer_gold.write.mode("overwrite").saveAsTable("shoplive.gold.customers")
print(f"shoplive.gold.customers: {customer_gold.count():,} rows")
display(customer_gold.limit(10))

In [0]:

customer_segments = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.datediff(F.current_date(), F.max("order_date")).alias("recency"),
        F.count("order_id").alias("frequency"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("monetary")
    )
    .withColumn(
        "segment",
        F.when((F.col("recency") <= 30) & (F.col("frequency") >= 10) & (F.col("monetary") >= 50000), "VIP")
        .when((F.col("recency") <= 30) & (F.col("frequency") >= 5) & (F.col("monetary") >= 10000), "Loyal")
        .when((F.col("recency") > 30) & (F.col("recency") <= 90) & (F.col("frequency") >= 3), "At Risk")
        .when(F.col("recency") > 90, "Churned")
        .otherwise("New")
    )
    .select("customer_id", "recency", "frequency", "monetary", "segment")
)

customer_segments.write.mode("overwrite").saveAsTable("shoplive.gold.customer_segments")
print(f"shoplive.gold.customer_segments: {customer_segments.count():,} rows")
display(customer_segments.limit(10))

In [0]:
product_performance = (
    orders_products_df
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity_sold"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_revenue"),
        F.round(F.sum(F.col("quantity") * F.col("unit_cost")), 2).alias("total_cost")
    )
    .withColumn("total_profit", F.round(F.col("total_revenue") - F.col("total_cost"), 2))
    .withColumn("profit_margin", F.round((F.col("total_profit") / F.col("total_revenue")) * 100, 2))
    .withColumn("average_selling_price", F.round(F.col("total_revenue") / F.col("total_quantity_sold"), 2))
    .select(
        "product_id", "product_name", "category",
        "total_orders", "total_quantity_sold", "total_revenue",
        "total_cost", "total_profit", "profit_margin", "average_selling_price"
    )
)

product_performance.write.mode("overwrite").saveAsTable("shoplive.gold.product_performance")
print(f"shoplive.gold.product_performance: {product_performance.count():,} rows")
display(product_performance.limit(10))

In [0]:
category_performance = (
    orders_products_df
    .groupBy("category")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_revenue"),
        F.round(F.sum(F.col("quantity") * F.col("unit_cost")), 2).alias("total_cost")
    )
    .withColumn("total_profit", F.round(F.col("total_revenue") - F.col("total_cost"), 2))
    .withColumn("profit_margin", F.round((F.col("total_profit") / F.col("total_revenue")) * 100, 2))
    .withColumn("average_order_value", F.round(F.col("total_revenue") / F.col("total_orders"), 2))
    .select(
        "category", "total_orders", "total_quantity", "total_revenue",
        "total_cost", "total_profit", "profit_margin", "average_order_value"
    )
)

category_performance.write.mode("overwrite").saveAsTable("shoplive.gold.category_performance")
print(f"shoplive.gold.category_performance: {category_performance.count():,} rows")
display(category_performance)

In [0]:
daily_category_sales = (
    orders_products_df
    .groupBy("order_date", "category")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_revenue"),
        F.round(F.sum(F.col("quantity") * (F.col("unit_price") - F.col("unit_cost"))), 2).alias("total_profit")
    )
    .select("order_date", "category", "total_orders", "total_quantity", "total_revenue", "total_profit")
    .orderBy("order_date", "category")
)

daily_category_sales.write.mode("overwrite").saveAsTable("shoplive.gold.daily_category_sales")
print(f"shoplive.gold.daily_category_sales: {daily_category_sales.count():,} rows")
display(daily_category_sales.limit(10))

In [0]:
order_status_summary = (
    orders_df
    .groupBy("status")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_revenue")
    )
    .select("status", "order_count", "total_quantity", "total_revenue")
)

order_status_summary.write.mode("overwrite").saveAsTable("shoplive.gold.order_status_summary")
print(f"shoplive.gold.order_status_summary: {order_status_summary.count():,} rows")
display(order_status_summary)